**XGBoost + GridSearch**

In [1]:
from sklearn.metrics import root_mean_squared_error


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, learning_curve
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error


print("📂 Chargement des données...")

try:
    df = pd.read_csv(r"C:\Users\july.rabot\Documents\Projet_Occas\voitures_ia3.csv")
    df.dropna(inplace=True)
    
    print(f"📊 Taille avant encodage : {df.shape}")
    
    # df = pd.get_dummies(df, drop_first=True, dtype=int)
    
    print(f"💥 Taille après encodage : {df.shape} (C'est normal que ça explose !)")
    
except FileNotFoundError:
    print("❌ ERREUR : Fichier 'voitures_ia_sans_get.csv' introuvable.")
    raise

ModuleNotFoundError: No module named 'xgboost'

In [ ]:
df
# X = df
X = df.drop(columns=["Price_EUR"])                    # Uniquement les variables d’entrée
y = df["Price_EUR"]    

In [ ]:
import numpy as np
import pandas as pd                                 # Data manipulation with DataFrame
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline  
from sklearn.impute import SimpleImputer              # Fill missing values (imputation)


def Model_Pipeline():
    print("\n[4) SÉLECTION DU MODÈLE + PRÉTRAITEMENT]")

    

    numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()   # Liste des colonnes numériques
    categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()  # Liste des colonnes catégorielles

    print("Numeric features:", numeric_features)           # Affiche les variables numériques
    print("Categorical features:", categorical_features)   # Affiche les variables catégorielles

    # Prétraitement pour les colonnes numériques : imputation par la médiane + mise à l’échelle
    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),     # Remplace les valeurs numériques manquantes par la médiane (calculée sur le TRAIN uniquement)
        ("scaler", StandardScaler())                       # Standardise les variables (ajusté sur le TRAIN uniquement)
    ])

    # Prétraitement pour les colonnes catégorielles : imputation par la valeur la plus fréquente + one-hot encoding
    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),  # Remplace les catégories manquantes par la plus fréquente (TRAIN uniquement)
        ("onehot", OneHotEncoder(handle_unknown="ignore"))     # Encode les catégories en one-hot (appris sur le TRAIN uniquement)
    ])

    # Combine le prétraitement numérique et catégoriel dans un seul objet
    preprocess = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),     # Applique le pipeline numérique aux colonnes numériques
            ("cat", categorical_transformer, categorical_features)  # Applique le pipeline catégoriel aux colonnes catégorielles
        ]
    )

    # print(preprocess)
    # print(X)
    
    return preprocess


# transformer = ColumnTransformer(transformers=[('Fuel_Type', OneHotEncoder(handle_unknown='ignore'), ['Fuel_Type'])], remainder="passthrough")

# df2 = transformer.fit_transform(X)
# print(df2)


In [ ]:
Model_Pipeline()


[4) SÉLECTION DU MODÈLE + PRÉTRAITEMENT]
Numeric features: ['Year', 'Kilometers_Driven', 'Mileage', 'Engine', 'Power', 'Seats']
Categorical features: ['Location', 'Fuel_Type', 'Transmission', 'Owner_Type', 'Marque', 'Modele']


,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

In [ ]:
preprocess = Model_Pipeline()


[4) SÉLECTION DU MODÈLE + PRÉTRAITEMENT]
Numeric features: ['Year', 'Kilometers_Driven', 'Mileage', 'Engine', 'Power', 'Seats']
Categorical features: ['Location', 'Fuel_Type', 'Transmission', 'Owner_Type', 'Marque', 'Modele']


In [ ]:
def split(self, X, y):
    X_train, X_test, y_train, y_test = train_test_split(  # Séparation en jeu d’entraînement et de test
        X, y,                                             # Variables d’entrée et variable cible
        test_size=0.2,                                    # 20% pour le test
        random_state=42                                   # Pour la reproductibilité
    )
    return X_train, X_test, y_train, y_test

In [ ]:
print("\n[5) MODEL TRAINING]")

X_train, X_test, y_train, y_test = train_test_split(  # Séparation en jeu d’entraînement et de test
    X, y,                                             # Variables d’entrée et variable cible
    test_size=0.2,                                    # 20% pour le test
    random_state=42                                   # Pour la reproductibilité
)

print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")  # Affiche les tailles des jeux de données

# A) Modèle de régression linéaire multivariable (baseline)
linreg_model = Pipeline(steps=[
    ("preprocess", preprocess),                        # Prétraitement (appris uniquement sur le train)
    ("model", XGBRegressor())                      # Modèle de régression linéaire
])

print("Colonne X:", list(X.columns))
print("Price_Eur dans X ?", "Price_EUR" in X.columns)
print("Colonne df :", list(df.columns))

linreg_model.fit(X_train, y_train)                     # Entraîne le pipeline uniquement sur le jeu d’entraînement
print("Linear Regression trained.")                    # Confirme la fin de l’entraînement


[5) MODEL TRAINING]
Train size: (4464, 12), Test size: (1116, 12)
Colonne X: ['Location', 'Year', 'Kilometers_Driven', 'Fuel_Type', 'Transmission', 'Owner_Type', 'Mileage', 'Engine', 'Power', 'Seats', 'Marque', 'Modele']
Price_Eur dans X ? False
Colonne df : ['Location', 'Year', 'Kilometers_Driven', 'Fuel_Type', 'Transmission', 'Owner_Type', 'Mileage', 'Engine', 'Power', 'Seats', 'Marque', 'Modele', 'Price_EUR']
Linear Regression trained.


In [ ]:
import joblib
import pandas as pd
import numpy as np

pack_de_survie = {
    'modele': linreg_model,      
    'colonnes': X_train.columns 
}

joblib.dump(pack_de_survie, 'mon_ia_voitures_v3.pkl')

print("Sauvegardée dans 'mon_ia_voitures_v1.pk2' !")

Sauvegardée dans 'mon_ia_voitures_v1.pk2' !


In [ ]:
def Model_Regression_linear(self, X_train, y_train, preprocess):
    # A) Modèle de régression linéaire multivariable (baseline)
    linreg_model = Pipeline(steps=[
        ("preprocess", preprocess),                        # Prétraitement (appris uniquement sur le train)
        ("model", (XGBRegressor))                      # Modèle de régression linéaire
    ])

    linreg_model = linreg_model.fit(X_train, y_train)                     # Entraîne le pipeline uniquement sur le jeu d’entraînement
    print("Linear Regression trained.")  
        
    return linreg_model                  # Confirme la fin de l’entraînement

In [ ]:
# Pipe = Model_Regression_linear()

TypeError: Model_Regression_linear() missing 4 required positional arguments: 'self', 'X_train', 'y_train', and 'preprocess'

In [ ]:
def Eval(model, X_test, y_test, name="model"):

    preds = model.predict(X_test)                      # Prédictions sur les données de test

    mae = mean_absolute_error(y_test, preds)           # Calcule le MAE
    rmse = np.sqrt(mean_squared_error(y_test, preds))  # Calcule le RMSE
    r2 = r2_score(y_test, preds)                       # Calcule le score R²

    print(f"\n--- {name} ---")                         # Affiche le nom du modèle
    print(f"MAE : {mae:.4f}")                          # Affiche le MAE
    print(f"RMSE: {rmse:.4f}")                         # Affiche le RMSE
    print(f"R^2 : {r2:.4f}")                           # Affiche le R²

    return mae, rmse, r2, preds                               # Retourne les métriques

def Graph_Metrics(self, y_test, preds, name):
        plt.figure()                                       # Nouvelle figure
        plt.scatter(y_test, preds, s=8)                    # Nuage de points réel vs prédit
        plt.title(f"{name}: Réel vs Prédit")               # Titre
        plt.xlabel("Actual")                               # Label axe X (valeurs réelles)
        plt.ylabel("Predicted")                            # Label axe Y (valeurs prédites)
        plt.show()                                         # Affiche le graphique

In [ ]:
    # Utilisation

# Évaluation de la régression linéaire
linreg_scores = Eval(                   # Évalue le modèle de base
    linreg_model,                                      # Pipeline
    X_test,                                            # Données de test (features)
    y_test,                                            # Données de test (cible)
    "Linear Regression"                                # Nom du modèle
)


--- Linear Regression ---
MAE : 1323.0549
RMSE: 2474.2623
R^2 : 0.9352


In [ ]:
X.head()

,Location,Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,Marque,Modele,Price_EUR
0,Mumbai,2010,72000,CNG,Manual,First,19.00,998.0,58,5,Maruti,Wagon R LXI CNG,1925
1,Pune,2015,41000,Diesel,Manual,First,19.67,1582.0,126,5,Hyundai,Creta 1.6 CRDi SX Option,13750
2,Chennai,2011,46000,Petrol,Manual,First,18.20,1199.0,88,5,Honda,Jazz V,4950
3,Chennai,2012,87000,Diesel,Manual,First,20.77,1248.0,88,7,Maruti,Ertiga VDI,6600
4,Coimbatore,2013,40670,Diesel,Automatic,Second,15.20,1968.0,140,5,Audi,A4 New 2.0 TDI Multitronic,19514


In [ ]:
df

array([[1.0, 0.0, 0.0, ..., 'Maruti', 'Wagon R LXI CNG', 1925],
       [1.0, 0.0, 0.0, ..., 'Hyundai', 'Creta 1.6 CRDi SX Option', 13750],
       [1.0, 0.0, 0.0, ..., 'Honda', 'Jazz V', 4950],
       ...,
       [1.0, 0.0, 0.0, ..., 'Mahindra', 'Xylo D4 BSIV', 3190],
       [1.0, 0.0, 0.0, ..., 'Maruti', 'Wagon R VXI', 2915],
       [1.0, 0.0, 0.0, ..., 'Chevrolet', 'Beat Diesel', 2750]],
      shape=(5580, 34), dtype=object)

In [ ]:
df.drop('Modele', axis=1, inplace=True)

In [ ]:
df.columns

Index(['Location', 'Year', 'Kilometers_Driven', 'Fuel_Type', 'Transmission',
       'Owner_Type', 'Mileage', 'Engine', 'Power', 'Seats', 'Marque', 'Modele',
       'Price_EUR'],
      dtype='str')

In [ ]:
df.Location.unique()

<StringArray>
[    'Mumbai',       'Pune',    'Chennai', 'Coimbatore',  'Hyderabad',
     'Jaipur',      'Kochi',    'Kolkata',      'Delhi',  'Bangalore',
  'Ahmedabad']
Length: 11, dtype: str

In [ ]:


X = df.drop(columns=['Price_EUR'])
y = df['Price_EUR']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("\n🔥 Démarrage du GridSearch... (Ça va chauffer avec toutes ces colonnes !)")

param_grid_xgb = {
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1],
    'max_depth': [3, 5, 7]
}

xgb = XGBRegressor(random_state=42)

grid = GridSearchCV(xgb, param_grid_xgb, cv=3, scoring='r2', n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print(f"\n✅ Meilleur modèle trouvé : {grid.best_params_}")


# ÉVALUATION & SCORES

predictions = best_model.predict(X_test)
r2 = r2_score(y_test, predictions)
mae = mean_absolute_error(y_test, predictions)
rmse = root_mean_squared_error(y_test, predictions)

print("\n" + "="*40)
print("🏆 RÉSULTATS (VERSION GET_DUMMIES)")
print("="*40)
print(f"⭐ Précision (R2)       : {r2:.2%}")
print(f"💰 Erreur Moyenne (MAE) : {int(mae)} €")
print(f"📏 Erreur RMSE          : {int(rmse)} €")
print(rmse)


# GRAPHIQUES (Nuage + Courbes)

print("\n📊 Génération des graphiques...")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Graphique Nuage de Points
sns.scatterplot(x=y_test, y=predictions, alpha=0.5, color="#673AB7", ax=axes[0])
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2)
axes[0].set_title(f"Réalité vs Prédictions\nR2 = {r2:.2%}", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Vrai Prix")
axes[0].set_ylabel("Prix Prédit")
axes[0].grid(True, linestyle='--', alpha=0.5)

# Graphique Learning Curve
train_sizes, train_scores, val_scores = learning_curve(
    best_model, X_train, y_train, cv=3, scoring='r2', n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 5)
)

train_mean = np.mean(train_scores, axis=1)
val_mean = np.mean(val_scores, axis=1)

axes[1].plot(train_sizes, train_mean, 'o-', color="#C2185B", label="Entraînement")
axes[1].plot(train_sizes, val_mean, 'o-', color="#4A148C", linewidth=2, label="Validation (Réalité)")
axes[1].fill_between(train_sizes, train_mean, val_mean, color="#E1BEE7", alpha=0.3)
axes[1].set_title("Progression de l'apprentissage", fontsize=12)
axes[1].set_xlabel("Nombre de voitures")
axes[1].set_ylabel("Score R2")
axes[1].legend()
axes[1].grid(True, linestyle='--', alpha=0.5)
axes[1].set_ylim(0.8, 1.01)

plt.tight_layout()
plt.show()

📂 Chargement des données...

🔥 Démarrage du GridSearch... (Ça va chauffer avec toutes ces colonnes !)
Fitting 3 folds for each of 12 candidates, totalling 36 fits


ValueError: 
All the 36 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
36 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\july.rabot\Documents\Projet_Occas\occas\Lib\site-packages\xgboost\data.py", line 408, in pandas_feature_info
    new_feature_types.append(_pandas_dtype_mapper[dtype.name])
                             ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^
KeyError: 'str'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "c:\Users\july.rabot\Documents\Projet_Occas\occas\Lib\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\july.rabot\Doc

In [ ]:
diff = y_test - predictions
rmsee = root_mean_squared_error(y_test, predictions)
print(rmsee)

2429.361083984375


In [ ]:
# 1. On calcule le prix moyen d'une voiture dans ton test
prix_moyen = y_test.mean()

# 2. On divise ton erreur par ce prix moyen
rmse_percentage = (rmsee / prix_moyen) * 100

print(f"💰 Prix moyen d'une voiture : {prix_moyen:,.2f} €")
print(f"📊 Ton erreur en pourcentage : {rmse_percentage:.2f}%")

💰 Prix moyen d'une voiture : 9,866.13 €
📊 Ton erreur en pourcentage : 24.62%


**Tests**

In [ ]:
print("\n🎮 Lancement du simulateur... À toi de jouer !")

# 👇 TES VALEURS ICI 👇
ma_voiture = {
    'Name': 'BMW',           
    'Location': 'Mumbai',     
    'Year': 2018,             
    'Kilometers_Driven': 2500,
    'Fuel_Type': 'Petrol',    
    'Transmission': 'Automatic',  
    'Owner_Type': 'Second',    
    'Mileage': 18.0,          
    'Engine': 2995.0,         
    'Power': 350.0,            
    'Seats': 5.0,             
    'New_Price': 0         
}

try:
    # 1. On crée le tableau brut
    df_simu = pd.DataFrame([ma_voiture])
    
    # 2. On applique le MÊME get_dummies que pour l'entraînement
    df_simu = pd.get_dummies(df_simu, drop_first=True, dtype=int)

    # 3. L'ALIGNEMENT DES PLANÈTES (Crucial !)
    # Le modèle veut 242 colonnes, mais df_simu en a peut-être que 10.
    # On force df_simu à avoir les mêmes colonnes que X_train.
    # On remplit les trous (fill_value) avec des 0.
    df_simu = df_simu.reindex(columns=X_train.columns, fill_value=0)

    # 4. Prédiction
    prix_estime = best_model.predict(df_simu)[0]
    print("\n" + "💸" * 15)
    print(f"  VOITURE : {ma_voiture['Name']} {ma_voiture['Year']} ({ma_voiture['Transmission']})")
    print(f"  ESTIMATION : {int(prix_estime):,} €")
    print("💸" * 15)

except Exception as e:
    print("❌ Erreur dans la simulation :", e)


🎮 Lancement du simulateur... À toi de jouer !

💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸
  VOITURE : BMW 2018 (Automatic)
  ESTIMATION : 50,845 €
💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸


In [ ]:
print("\n🎮 Lancement du simulateur corrigé... À toi de jouer !")

# 👇 TES VALEURS ICI 👇
ma_voiture = {
    'Name': 'BMW',              
    'Location': 'Mumbai',       # J'ai remis Mumbai pour être sûr
    'Year': 2018,               
    'Kilometers_Driven': 2500,  
    'Fuel_Type': 'Petrol',      
    'Transmission': 'Automatic', # <--- TESTE 'Manual' PUIS 'Automatic' ICI
    'Owner_Type': 'Second',      
    'Mileage': 18.0,            
    'Engine': 2995.0,           
    'Power': 350.0,             
    'Seats': 5.0,               
    'New_Price': 0              
}

try:
    # 1. On crée le tableau brut
    df_simu = pd.DataFrame([ma_voiture])
    
    # 2. CORRECTION ICI : On met drop_first=FALSE pour ne pas perdre l'info sur une seule ligne
    df_simu = pd.get_dummies(df_simu, drop_first=False, dtype=int)

    # 3. L'ALIGNEMENT DES PLANÈTES
    # On force df_simu à avoir les mêmes colonnes que le modèle (X_train)
    # C'est ici que la magie opère : si la colonne 'Transmission_Manual' manque,
    # le reindex va la créer proprement.
    df_simu = df_simu.reindex(columns=X_train.columns, fill_value=0)

    # 4. Prédiction
    prix_estime = best_model.predict(df_simu)[0]

    print("\n" + "💸" * 15)
    print(f"  VOITURE : {ma_voiture['Name']} {ma_voiture['Year']} ({ma_voiture['Transmission']})")
    print(f"  ESTIMATION : {int(prix_estime):,} €")
    print("💸" * 15)

except Exception as e:
    print("❌ Erreur dans la simulation :", e)


🎮 Lancement du simulateur corrigé... À toi de jouer !

💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸
  VOITURE : BMW 2018 (Automatic)
  ESTIMATION : 50,634 €
💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸


In [ ]:
print("\n🎮 Lancement du simulateur corrigé... À toi de jouer !")

# 👇 TES VALEURS ICI 👇
ma_voiture = {
    'Name': 'BMW',              
    'Location': 'Mumbai',       # J'ai remis Mumbai pour être sûr
    'Year': 2018,               
    'Kilometers_Driven': 2500,  
    'Fuel_Type': 'Petrol',      
    'Transmission': 'Manual', # <--- TESTE 'Manual' PUIS 'Automatic' ICI
    'Owner_Type': 'Second',      
    'Mileage': 18.0,            
    'Engine': 2995.0,           
    'Power': 350.0,             
    'Seats': 5.0,               
    'New_Price': 0              
}

try:
    # 1. On crée le tableau brut
    df_simu = pd.DataFrame([ma_voiture])
    
    # 2. CORRECTION ICI : On met drop_first=FALSE pour ne pas perdre l'info sur une seule ligne
    df_simu = pd.get_dummies(df_simu, drop_first=False, dtype=int)

    # 3. L'ALIGNEMENT DES PLANÈTES
    # On force df_simu à avoir les mêmes colonnes que le modèle (X_train)
    # C'est ici que la magie opère : si la colonne 'Transmission_Manual' manque,
    # le reindex va la créer proprement.
    df_simu = df_simu.reindex(columns=X_train.columns, fill_value=0)

    # 4. Prédiction
    prix_estime = best_model.predict(df_simu)[0]

    print("\n" + "💸" * 15)
    print(f"  VOITURE : {ma_voiture['Name']} {ma_voiture['Year']} ({ma_voiture['Transmission']})")
    print(f"  ESTIMATION : {int(prix_estime):,} €")
    print("💸" * 15)

except Exception as e:
    print("❌ Erreur dans la simulation :", e)


🎮 Lancement du simulateur corrigé... À toi de jouer !

💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸
  VOITURE : BMW 2018 (Manual)
  ESTIMATION : 50,327 €
💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸


In [ ]:
print("\n🎮 Lancement du simulateur corrigé... À toi de jouer !")

# 👇 TES VALEURS ICI 👇
ma_voiture = {
    'Name': 'Maruti',           
    'Location': 'Mumbai',       
    'Year': 2016,               
    'Kilometers_Driven': 60000, # Elle a roulé
    'Fuel_Type': 'Petrol',      
    'Transmission': 'Manual',   # <--- TESTE 'Manual' PUIS 'Automatic'
    'Owner_Type': 'First',      
    'Mileage': 20.0,            
    'Engine': 1197.0,           
    'Power': 80.0,              # Petit moteur standard
    'Seats': 5.0,               
    'New_Price': 0             
}

try:
    # 1. On crée le tableau brut
    df_simu = pd.DataFrame([ma_voiture])
    
    # 2. CORRECTION ICI : On met drop_first=FALSE pour ne pas perdre l'info sur une seule ligne
    df_simu = pd.get_dummies(df_simu, drop_first=False, dtype=int)

    # 3. L'ALIGNEMENT DES PLANÈTES
    # On force df_simu à avoir les mêmes colonnes que le modèle (X_train)
    # C'est ici que la magie opère : si la colonne 'Transmission_Manual' manque,
    # le reindex va la créer proprement.
    df_simu = df_simu.reindex(columns=X_train.columns, fill_value=0)

    # 4. Prédiction
    prix_estime = best_model.predict(df_simu)[0]

    print("\n" + "💸" * 15)
    print(f"  VOITURE : {ma_voiture['Name']} {ma_voiture['Year']} ({ma_voiture['Transmission']})")
    print(f"  ESTIMATION : {int(prix_estime):,} €")
    print("💸" * 15)

except Exception as e:
    print("❌ Erreur dans la simulation :", e)


🎮 Lancement du simulateur corrigé... À toi de jouer !

💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸
  VOITURE : Maruti 2016 (Manual)
  ESTIMATION : 5,459 €
💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸


In [ ]:
print("\n🎮 Lancement du simulateur corrigé... À toi de jouer !")

# 👇 TES VALEURS ICI 👇
ma_voiture = {
    'Name': 'Maruti',           
    'Location': 'Mumbai',       
    'Year': 2016,               
    'Kilometers_Driven': 60000, # Elle a roulé
    'Fuel_Type': 'Petrol',      
    'Transmission': 'Automatic',   # <--- TESTE 'Manual' PUIS 'Automatic'
    'Owner_Type': 'First',      
    'Mileage': 20.0,            
    'Engine': 1197.0,           
    'Power': 80.0,              # Petit moteur standard
    'Seats': 5.0,               
    'New_Price': 0             
}

try:
    # 1. On crée le tableau brut
    df_simu = pd.DataFrame([ma_voiture])
    
    # 2. CORRECTION ICI : On met drop_first=FALSE pour ne pas perdre l'info sur une seule ligne
    df_simu = pd.get_dummies(df_simu, drop_first=False, dtype=int)

    # 3. L'ALIGNEMENT DES PLANÈTES
    # On force df_simu à avoir les mêmes colonnes que le modèle (X_train)
    # C'est ici que la magie opère : si la colonne 'Transmission_Manual' manque,
    # le reindex va la créer proprement.
    df_simu = df_simu.reindex(columns=X_train.columns, fill_value=0)

    # 4. Prédiction
    prix_estime = best_model.predict(df_simu)[0]

    print("\n" + "💸" * 15)
    print(f"  VOITURE : {ma_voiture['Name']} {ma_voiture['Year']} ({ma_voiture['Transmission']})")
    print(f"  ESTIMATION : {int(prix_estime):,} €")
    print("💸" * 15)

except Exception as e:
    print("❌ Erreur dans la simulation :", e)


🎮 Lancement du simulateur corrigé... À toi de jouer !

💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸
  VOITURE : Maruti 2016 (Automatic)
  ESTIMATION : 6,131 €
💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸


**Sauvegarde du Modèle**

In [ ]:
# import joblib
# import pandas as pd
# import numpy as np

# pack_de_survie = {
#     'modele': best_model,      
#     'colonnes': X_train.columns 
# }

# joblib.dump(pack_de_survie, 'mon_ia_voitures_v2.pkl')

# print("Sauvegardée dans 'mon_ia_voitures_v1.pk2' !")

Sauvegardée dans 'mon_ia_voitures_v1.pk2' !


**Crash Test**

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split

print("📂 Chargement du modèle sauvegardé et des données...")

# 1. ON CHARGE LE CERVEAU (Ton modèle entraîné)
try:
    # On charge le pack (Modèle + Liste des colonnes)
    pack = joblib.load('mon_ia_voitures_v2.pkl')
    best_model = pack['modele']
    colonnes_du_modele = pack['colonnes']
    print("✅ Modèle chargé avec succès !")
except:
    print("❌ ERREUR : Je ne trouve pas 'mon_ia_voitures_get_dummies.pkl'. As-tu bien lancé la sauvegarde ?")
    raise

# 2. ON CHARGE LE FICHIER LISIBLE
# C'est celui qui contient "Marque", "Modele", "Year"...
df_lisible = pd.read_csv('C:\\Users\\july.rabot\\Documents\\Projet_Occas\\Data\\voitures_ia_sans_get.csv')
df_lisible.dropna(inplace=True)

# 3. ON RETROUVE LE TEST SET (L'étape cruciale)
# On doit refaire le "get_dummies" et le "split" exactement comme avant
# pour savoir quelles lignes ont servi au test.
df_maths = pd.get_dummies(df_lisible, drop_first=True, dtype=int)

# On aligne les colonnes (au cas où)
df_maths = df_maths.reindex(columns=colonnes_du_modele.tolist() + ['Price_EUR'], fill_value=0)

X = df_maths.drop(columns=['Price_EUR'])
y = df_maths['Price_EUR']

# Le fameux random_state=42 nous garantit qu'on retombe sur les MÊMES voitures
_, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. ON PRÉDIT (Sans réentraîner)
print("🔮 Calcul des prédictions sur les données de test...")
predictions = best_model.predict(X_test)

# 5. LE CRASH TEST (Assemblage)
print("\n" + "="*60)
print("💀 LE BÊTISIER FINAL : Analyse des pires erreurs")
print("="*60)

# On va chercher les infos lisibles grâce aux index de y_test
df_erreurs = df_lisible.loc[y_test.index].copy()

df_erreurs['Vrai_Prix'] = y_test
df_erreurs['Prix_IA'] = predictions
df_erreurs['Ecart_Absolu'] = (df_erreurs['Vrai_Prix'] - df_erreurs['Prix_IA']).abs()

# Top 10 des erreurs
top_pires = df_erreurs.sort_values(by='Ecart_Absolu', ascending=False).head(10)

for index, row in top_pires.iterrows():
    # Reconstruction du nom (Marque + Modèle)
    marque = row.get('Marque', 'MarqueInconnue')
    modele = row.get('Modele', '') # Vide si pas de colonne modèle
    nom_complet = f"{marque} {modele}"
    
    # Gestion des autres colonnes (si elles ont changé de nom)
    fuel = row.get('Fuel_Type', row.get('Fuel', '?'))
    boite = row.get('Transmission', '?')
    
    percent = (row['Ecart_Absolu'] / row['Vrai_Prix']) * 100
    
    print(f"🚗 {nom_complet} ({row['Year']})")
    print(f"   📍 {row['Location']} | ⛽ {fuel} | ⚙️ {boite}")
    print(f"   💰 Vrai Prix : {int(row['Vrai_Prix']):,} €")
    print(f"   🤖 Prix IA   : {int(row['Prix_IA']):,} €")
    
    diff = int(row['Ecart_Absolu'])
    if row['Prix_IA'] > row['Vrai_Prix']:
        print(f"   ❌ Erreur    : +{diff:,} € (SURESTIMÉ - Trop cher !)")
    else:
        print(f"   ❌ Erreur    : -{diff:,} € (SOUS-ESTIMÉ - L'affaire du siècle ?)")
        
    print(f"   📉 Marge d'erreur : {int(percent)}%")
    print("-" * 50)

📂 Chargement du modèle sauvegardé et des données...
✅ Modèle chargé avec succès !
🔮 Calcul des prédictions sur les données de test...

💀 LE BÊTISIER FINAL : Analyse des pires erreurs
🚗 Land Rover Discovery Sport TD4 HSE (2019)
   📍 Coimbatore | ⛽ ? | ⚙️ ?
   💰 Vrai Prix : 61,270 €
   🤖 Prix IA   : 41,672 €
   ❌ Erreur    : -19,597 € (SOUS-ESTIMÉ - L'affaire du siècle ?)
   📉 Marge d'erreur : 31%
--------------------------------------------------
🚗 Jaguar XE Portfolio (2017)
   📍 Mumbai | ⛽ ? | ⚙️ ?
   💰 Vrai Prix : 37,950 €
   🤖 Prix IA   : 54,678 €
   ❌ Erreur    : +16,728 € (SURESTIMÉ - Trop cher !)
   📉 Marge d'erreur : 44%
--------------------------------------------------
🚗 Land Rover Discovery 4 TDV6 SE (2013)
   📍 Delhi | ⛽ ? | ⚙️ ?
   💰 Vrai Prix : 37,400 €
   🤖 Prix IA   : 21,088 €
   ❌ Erreur    : -16,311 € (SOUS-ESTIMÉ - L'affaire du siècle ?)
   📉 Marge d'erreur : 43%
--------------------------------------------------
🚗 Nissan Teana XV (2015)
   📍 Mumbai | ⛽ ? | ⚙️ ?
   💰 V

La première : C'est soit une erreur de saisie dans le fichier d'origine (le vendeur a oublié un zéro ?), soit la voiture est une épave accidentée.

La deuxième : Les Land Rover sont des voitures très difficiles à estimer. Elles décotent très vite (car l'entretien coûte cher), mais certains modèles de passionnés gardent une valeur folle. Ton IA, qui a surtout appris sur des Maruti et des Hyundai, a du mal à saisir la "valeur passion" d'un gros 4x4 anglais. Elle l'a traité comme une "vielle voiture de 2013" classique.

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split

print("📂 Chargement du modèle sauvegardé et des données...")

# 1. ON CHARGE LE CERVEAU (Ton modèle entraîné)
try:
    # On charge le pack (Modèle + Liste des colonnes)
    pack = joblib.load('mon_ia_voitures_v2.pkl')
    best_model = pack['modele']
    colonnes_du_modele = pack['colonnes']
    print("✅ Modèle chargé avec succès !")
except:
    print("❌ ERREUR : Je ne trouve pas 'mon_ia_voitures_get_dummies.pk2'. As-tu bien lancé la sauvegarde ?")
    raise

# 2. ON CHARGE LE FICHIER LISIBLE
# C'est celui qui contient "Marque", "Modele", "Year"...
df_lisible = pd.read_csv('C:\\Users\\july.rabot\\Documents\\Projet_Occas\\Data\\voitures_ia_sans_get.csv')
df_lisible.dropna(inplace=True)

# 3. ON RETROUVE LE TEST SET (L'étape cruciale)
# On doit refaire le "get_dummies" et le "split" exactement comme avant
# pour savoir quelles lignes ont servi au test.
df_maths = pd.get_dummies(df_lisible, drop_first=True, dtype=int)

# On aligne les colonnes (au cas où)
df_maths = df_maths.reindex(columns=colonnes_du_modele.tolist() + ['Price_EUR'], fill_value=0)

X = df_maths.drop(columns=['Price_EUR'])
y = df_maths['Price_EUR']

# Le fameux random_state=42 nous garantit qu'on retombe sur les MÊMES voitures
_, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. ON PRÉDIT (Sans réentraîner)
print("🔮 Calcul des prédictions sur les données de test...")
predictions = best_model.predict(X_test)

# 5. LE CRASH TEST (Assemblage)
print("\n" + "="*60)
print("💀 LE BÊTISIER FINAL : Analyse des pires erreurs")
print("="*60)

# On va chercher les infos lisibles grâce aux index de y_test
df_erreurs = df_lisible.loc[y_test.index].copy()

df_erreurs['Vrai_Prix'] = y_test
df_erreurs['Prix_IA'] = predictions
df_erreurs['Ecart_Absolu'] = (df_erreurs['Vrai_Prix'] - df_erreurs['Prix_IA']).abs()

# Top 10 des erreurs
top_pires = df_erreurs.sort_values(by='Ecart_Absolu', ascending=False).head(10)

for index, row in top_pires.iterrows():
    # Reconstruction du nom (Marque + Modèle)
    marque = row.get('Marque', 'MarqueInconnue')
    modele = row.get('Modele', '') # Vide si pas de colonne modèle
    nom_complet = f"{marque} {modele}"
    
    # Gestion des autres colonnes (si elles ont changé de nom)
    fuel = row.get('Fuel_Type', row.get('Fuel', '?'))
    boite = row.get('Transmission', '?')
    
    percent = (row['Ecart_Absolu'] / row['Vrai_Prix']) * 100
    
    print(f"🚗 {nom_complet} ({row['Year']})")
    print(f"   📍 {row['Location']} | ⛽ {fuel} | ⚙️ {boite}")
    print(f"   💰 Vrai Prix : {int(row['Vrai_Prix']):,} €")
    print(f"   🤖 Prix IA   : {int(row['Prix_IA']):,} €")
    
    diff = int(row['Ecart_Absolu'])
    if row['Prix_IA'] > row['Vrai_Prix']:
        print(f"   ❌ Erreur    : +{diff:,} € (SURESTIMÉ - Trop cher !)")
    else:
        print(f"   ❌ Erreur    : -{diff:,} € (SOUS-ESTIMÉ - L'affaire du siècle ?)")
        
    print(f"   📉 Marge d'erreur : {int(percent)}%")
    print("-" * 50)

📂 Chargement du modèle sauvegardé et des données...
✅ Modèle chargé avec succès !
🔮 Calcul des prédictions sur les données de test...

💀 LE BÊTISIER FINAL : Analyse des pires erreurs
🚗 Land Rover Discovery Sport TD4 HSE (2019)
   📍 Coimbatore | ⛽ ? | ⚙️ ?
   💰 Vrai Prix : 61,270 €
   🤖 Prix IA   : 41,672 €
   ❌ Erreur    : -19,597 € (SOUS-ESTIMÉ - L'affaire du siècle ?)
   📉 Marge d'erreur : 31%
--------------------------------------------------
🚗 Jaguar XE Portfolio (2017)
   📍 Mumbai | ⛽ ? | ⚙️ ?
   💰 Vrai Prix : 37,950 €
   🤖 Prix IA   : 54,678 €
   ❌ Erreur    : +16,728 € (SURESTIMÉ - Trop cher !)
   📉 Marge d'erreur : 44%
--------------------------------------------------
🚗 Land Rover Discovery 4 TDV6 SE (2013)
   📍 Delhi | ⛽ ? | ⚙️ ?
   💰 Vrai Prix : 37,400 €
   🤖 Prix IA   : 21,088 €
   ❌ Erreur    : -16,311 € (SOUS-ESTIMÉ - L'affaire du siècle ?)
   📉 Marge d'erreur : 43%
--------------------------------------------------
🚗 Nissan Teana XV (2015)
   📍 Mumbai | ⛽ ? | ⚙️ ?
   💰 V

In [ ]:
df.info()